In [ ]:
# SHARED — support-ticket domain + Pydantic AI agent + FunctionModel helpers.
# This is the system under test. NO API key, NO network.
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Optional

import pydantic_ai
from pydantic import BaseModel, Field, ValidationError
from pydantic_ai import Agent, ModelResponse, RunContext, TextPart, ToolCallPart, capture_run_messages
from pydantic_ai.models.function import AgentInfo, FunctionModel
from pydantic_ai.models.test import TestModel

# --- prevent accidental real API calls (slide 12.9) ---
pydantic_ai.models.ALLOW_MODEL_REQUESTS = False

# --- domain objects (same as module-11) ---
@dataclass
class SupportTicket:
    id: int
    subject: str
    category: str          # "Billing" | "Technical" | "Account"
    priority: str          # "low" | "normal" | "high" | "urgent"
    is_open: bool = True
    tags: list[str] = field(default_factory=list)

SAMPLE_TICKETS = [
    SupportTicket(1, "Can't log in after password reset", "Account", "high"),
    SupportTicket(2, "Invoice double-charged this month",  "Billing", "urgent"),
    SupportTicket(3, "How do I export my data?",            "Technical", "low"),
    SupportTicket(4, "App crashes on upload",               "Technical", "high"),
    SupportTicket(5, "Refund not received",                 "Billing", "normal", is_open=False),
]

# --- the Pydantic AI agent (replaces the hand-rolled TicketAgent) ---
SYSTEM_PROMPT = ("You are a support-desk assistant. Use the tools to answer "
                 "questions about support tickets. Prefer one accurate tool call.")

ticket_agent = Agent(
    "test",                         # placeholder — overridden in every test
    deps_type=list[SupportTicket],
    instructions=SYSTEM_PROMPT,
)

@ticket_agent.tool
def list_open_tickets(ctx: RunContext[list[SupportTicket]]) -> list[dict]:
    """List all currently open tickets."""
    return [t.__dict__ for t in ctx.deps if t.is_open]

@ticket_agent.tool
def search_tickets(ctx: RunContext[list[SupportTicket]], term: str) -> list[dict]:
    """Find tickets whose subject contains a substring (case-insensitive)."""
    return [t.__dict__ for t in ctx.deps if term.lower() in t.subject.lower()]

@ticket_agent.tool
def count_by_priority(ctx: RunContext[list[SupportTicket]]) -> dict[str, int]:
    """Count tickets grouped by priority."""
    out: dict[str, int] = {}
    for t in ctx.deps:
        out[t.priority] = out.get(t.priority, 0) + 1
    return out

# --- structured output schema (Module 10 pattern) ---
class TicketQuery(BaseModel):
    category: Optional[str] = Field(default=None, description="Restrict to this category, or null for all.")
    priority: Optional[str] = Field(default=None, description="One of low|normal|high|urgent, or null.")
    open_only: bool = False
    subject_contains: Optional[str] = Field(default=None, description="Substring the subject must contain.")

# --- FunctionModel helpers — script the LLM's tool calls ---
def _force_tool_call(tool_name: str, **kwargs):
    """Return a FunctionModel callback that calls one tool, then answers."""
    def model_fn(messages, info):
        for msg in messages:
            for part in msg.parts:
                if part.part_kind == "tool-return":
                    return ModelResponse(parts=[TextPart("Done.")])
        return ModelResponse(parts=[ToolCallPart(tool_name, kwargs)])
    return model_fn

def _scripted_calls(tool_sequence: list[tuple[str, dict]], final_answer: str):
    """Return a FunctionModel callback that plays a sequence of tool calls."""
    def model_fn(messages, info):
        tool_returns = sum(
            1 for msg in messages for p in msg.parts if p.part_kind == "tool-return"
        )
        if tool_returns < len(tool_sequence):
            name, args = tool_sequence[tool_returns]
            return ModelResponse(parts=[ToolCallPart(name, args)])
        return ModelResponse(parts=[TextPart(final_answer)])
    return model_fn

# tiny in-notebook test harness (NOT pytest — just asserts you can watch run)
def check(label, cond):
    print(("PASS" if cond else "FAIL"), label)
    assert cond

print("Setup complete — ticket_agent ready with 3 tools.")

# Module 12 ⭐ — Code Along: Testing & validating AI

An LLM is **non-deterministic**: the same prompt yields a *distribution* of outputs. You can't assert on the prose. So we never test the wording — we test **four other things**, each a different class of check:

| Class | What it pins down | Needs an API key? |
|---|---|---|
| 1 · Tool tests | the deterministic Python the agent calls | no |
| 2 · Schema tests | the LLM's JSON validates against Pydantic | no |
| 3 · Loop tests | the orchestration — calls, steps, guards (mock the LLM) | no |
| 4 · Golden evals | locked behaviour on named cases | no (mocked) |

Assert on **shape, behaviour, substring, bounds** — never on the exact sentence. Three of the four classes need no key and run in the **same Day-3 CI**. Each cell below builds an agent, exercises it, and asserts with `check(...)`.

In [ ]:
# CLASS 1 — tool tests: the tools are plain Python, so test them like any function.
# With Pydantic AI, tools take RunContext as the first arg — you can't call them
# directly. Instead, force a specific tool call with FunctionModel and inspect
# the return value via capture_run_messages().

# search_tickets — force the tool with term="RESET", check it finds ticket 1
with ticket_agent.override(model=FunctionModel(_force_tool_call("search_tickets", term="RESET"))):
    with capture_run_messages() as msgs:
        ticket_agent.run_sync("find reset tickets", deps=SAMPLE_TICKETS)
tool_return = [p for msg in msgs for p in msg.parts if p.part_kind == "tool-return"][0]
data = tool_return.content
check("search is case-insensitive", data[0]["id"] == 1)

# search for a term that matches nothing
with ticket_agent.override(model=FunctionModel(_force_tool_call("search_tickets", term="quantum"))):
    with capture_run_messages() as msgs:
        ticket_agent.run_sync("find quantum", deps=SAMPLE_TICKETS)
tool_return = [p for msg in msgs for p in msg.parts if p.part_kind == "tool-return"][0]
check("search misses unrelated terms", tool_return.content == [])

# count_by_priority
with ticket_agent.override(model=FunctionModel(_force_tool_call("count_by_priority"))):
    with capture_run_messages() as msgs:
        ticket_agent.run_sync("count", deps=SAMPLE_TICKETS)
tool_return = [p for msg in msgs for p in msg.parts if p.part_kind == "tool-return"][0]
check("count_by_priority", tool_return.content == {"high": 2, "urgent": 1, "low": 1, "normal": 1})

# list_open_tickets — ticket 5 is closed, should be excluded
with ticket_agent.override(model=FunctionModel(_force_tool_call("list_open_tickets"))):
    with capture_run_messages() as msgs:
        ticket_agent.run_sync("show open", deps=SAMPLE_TICKETS)
tool_return = [p for msg in msgs for p in msg.parts if p.part_kind == "tool-return"][0]
open_ids = [t["id"] for t in tool_return.content]
check("closed ticket 5 is excluded", open_ids == [1, 2, 3, 4])

## Class 2 — schema tests

When the LLM returns JSON (the Module 10 extraction task), validate it against the **same Pydantic discipline from Day 2**. Good JSON parses into a typed object; malformed JSON raises a structured `ValidationError` you can catch — the model is your contract.

In [ ]:
# CLASS 2 — schema tests: validate the LLM's JSON against TicketQuery.
# Optional fields default cleanly when the model omits them.
check("optional fields default to None", TicketQuery().category is None)
check("open_only defaults False", TicketQuery().open_only is False)

# Valid JSON parses into a typed object.
q = TicketQuery.model_validate_json('{"category": "Billing", "open_only": true}')
check("valid JSON parses", q.category == "Billing" and q.open_only is True)

# Invalid payload — open_only must be a bool, not the string "maybe" — must raise.
raised = False
try:
    TicketQuery.model_validate({"open_only": "maybe"})
except ValidationError as e:
    raised = True
    print("caught ValidationError ->", e.errors()[0]["loc"], e.errors()[0]["type"])
check("bad type raises ValidationError", raised)

## Class 3 — loop tests

Script the LLM with **`FunctionModel`** — the Pydantic AI equivalent of Day-3's mock seam. `_scripted_calls` plays a sequence of tool calls followed by a final text answer. `capture_run_messages()` lets you inspect exactly which tools fired and in what order — assert on **behaviour**, never prose.

In [ ]:
# CLASS 3 — loop test: one tool call, then a final answer.
with ticket_agent.override(model=FunctionModel(
    _scripted_calls([("count_by_priority", {})],
                    "There are 2 high, 1 urgent, 1 low and 1 normal ticket.")
)):
    with capture_run_messages() as msgs:
        result = ticket_agent.run_sync("how many tickets by priority?", deps=SAMPLE_TICKETS)

tool_calls = [p.tool_name for msg in msgs for p in msg.parts if p.part_kind == "tool-call"]
check("exactly one tool call", tool_calls == ["count_by_priority"])

tool_returns = [p for msg in msgs for p in msg.parts if p.part_kind == "tool-return"]
check("tool result fed back into the loop",
      tool_returns[0].content["high"] == 2)

check("answer contains the key fact", "2 high" in result.output)

## Class 3b — safety nets

Two framework-level protections that Pydantic AI gives you for free:

1. **`ALLOW_MODEL_REQUESTS = False`** — any test that forgets to `override()` hits a hard error instead of a surprise API bill.
2. **`TestModel`** — auto-calls every registered tool with default-typed arguments (`str → "a"`, `int → 0`). A one-line smoke test that all your tools are callable.

In [ ]:
# ALLOW_MODEL_REQUESTS is already False (set in the shared cell).
# If we forget to override, the framework raises immediately:
raised = False
try:
    ticket_agent.run_sync("oops, no override", deps=SAMPLE_TICKETS)
except AttributeError:
    raised = True
    print("caught: model requests are blocked (no override set)")
check("ALLOW_MODEL_REQUESTS blocks accidental calls", raised)

# TestModel — smoke test: calls every registered tool with default args.
with ticket_agent.override(model=TestModel()):
    result = ticket_agent.run_sync("smoke test", deps=SAMPLE_TICKETS)
check("TestModel produces output", bool(result.output))
print("TestModel output:", result.output[:80], "...")

# Answer without tools: a FunctionModel that returns text immediately.
def just_answer(messages, info):
    return ModelResponse(parts=[TextPart("No tools needed, the answer is 42.")])

with ticket_agent.override(model=FunctionModel(just_answer)):
    result = ticket_agent.run_sync("nothing to do", deps=SAMPLE_TICKETS)
check("direct answer (no tool calls)", "42" in result.output)

## Class 4 — golden evals

A small JSON file of **named cases** locks behaviour: each case names a prompt, the tool sequence it should drive, and substrings the answer must contain. We **parametrize** over the cases — one loop, many checks. In real CI this lives in `tests/evals/golden_queries.json` and runs under the `eval` marker.

In [ ]:
# CLASS 4 — golden evals: the mocked agent loop driven over a table of golden cases.
# In real CI this lives in tests/evals/golden_queries.json and runs under @pytest.mark.eval.
GOLDEN = [
    {"id": "t-01", "prompt": "how many by priority?",
     "expected_tool_calls": ["count_by_priority"], "tool_args": [{}],
     "expected_answer_contains": ["high"]},
    {"id": "t-02", "prompt": "find login issues",
     "expected_tool_calls": ["search_tickets"], "tool_args": [{"term": "log in"}],
     "expected_answer_contains": ["log in"]},
]

ANSWERS = {
    "t-01": "By priority: 2 high, 1 urgent, 1 low, 1 normal.",
    "t-02": "Ticket 1 is about not being able to log in after a password reset.",
}

for case in GOLDEN:
    tool_sequence = list(zip(case["expected_tool_calls"], case["tool_args"]))

    with ticket_agent.override(model=FunctionModel(
        _scripted_calls(tool_sequence, ANSWERS[case["id"]])
    )):
        with capture_run_messages() as msgs:
            result = ticket_agent.run_sync(case["prompt"], deps=SAMPLE_TICKETS)

    got_tools = [p.tool_name for msg in msgs for p in msg.parts if p.part_kind == "tool-call"]
    check(f"[{case['id']}] tool sequence", got_tools == case["expected_tool_calls"])
    for needle in case["expected_answer_contains"]:
        check(f"[{case['id']}] answer contains {needle!r}", needle in result.output)

## Logfire — observability for the last rung

Tests catch bugs *before* deploy. **Logfire** catches them *after* — it's Pydantic's tracing platform, and it auto-instruments every Pydantic AI agent run. Two lines, no code changes to your agent or tests:

```python
logfire.configure(send_to_logfire=False)   # local console, no token
logfire.instrument_pydantic_ai()           # auto-trace every run_sync()
```

You get a nested trace: which tools fired, how long each took, what failed. Same data as `capture_run_messages()` in tests, but for production, in real time.

In [ ]:
# LOGFIRE — two lines to auto-trace every agent run.
# send_to_logfire=False → console output only, no token or account needed.
import logfire

logfire.configure(send_to_logfire=False)
logfire.instrument_pydantic_ai()

# Now run the same scripted agent — watch the trace appear in the output:
with ticket_agent.override(model=FunctionModel(
    _scripted_calls([("count_by_priority", {})],
                    "There are 2 high, 1 urgent, 1 low and 1 normal ticket.")
)):
    result = ticket_agent.run_sync("how many tickets by priority?", deps=SAMPLE_TICKETS)

print("\nAnswer:", result.output)
print("\n--- The indented lines above are Logfire spans ---")
print("Each span = one timed operation: agent run > model call > tool execution.")
print("In production, drop send_to_logfire=False and set LOGFIRE_TOKEN")
print("to stream these to the Logfire dashboard.")

## Recap

- **Test shape, behaviour, substring, bounds — never the prose.** The LLM is a distribution; the wording isn't a contract.
- **All four classes run without an API key** — `ALLOW_MODEL_REQUESTS = False` + `FunctionModel` / `TestModel` keep everything local and deterministic.
- These run in the **same Day-3 CI** you already built — same `pytest`, same green bar.
- `FunctionModel` is the Pydantic AI equivalent of Day-3's dependency-injection seam — same pattern, new framework.
- **Logfire** auto-traces every agent run — two lines, no code changes. Tests catch bugs before deploy; Logfire catches them after.

**→ Lab 12** writes these four classes as real `pytest` tests for the **`catalog_agent`** (Product domain) — same patterns, your spine.